# Session 24 — CI/CD Pipeline for Machine Learning with a Model Quality Gate

**Goal:** build a pipeline that retrains a model on every pull request and **refuses to
merge** unless the candidate model clears a fixed quality bar *and* does not regress
against the model currently in production — then watch one PR pass the gate and one PR
get blocked by it.

## What a quality gate adds to CI

Session 10 (`Implementing CI-CD Pipelines with GitHub Actions for MLOps`) sets up the
mechanics: a workflow triggers on push, installs dependencies, runs `pytest`, and the PR
goes green or red. That is ordinary software CI applied to an ML repo, and it catches
ordinary software failures — a syntax error, a broken import, a unit test that no longer
passes.

It does not catch the failure mode that actually matters here. A change to feature
engineering, a bumped scikit-learn version, or an "obviously harmless" hyperparameter
tweak can leave every test green while quietly making the model worse. Nothing in a
normal test suite knows what accuracy *was* yesterday.

A **model quality gate** closes that hole with two rules, and the second is the one that
does the real work:

1. **Absolute floor** — the candidate must beat fixed thresholds (accuracy ≥ 0.88,
   macro F1 ≥ 0.90). This stops something catastrophically broken from ever reaching
   production.
2. **No regression** — the candidate must also be within a small tolerance of, or better
   than, the **currently deployed** model's metrics. This is a ratchet: as production
   improves, the bar rises with it, and "still above the floor" stops being good enough.

The gate is a process running in CI that exits non-zero. Branch protection turns a
non-zero exit into a merge that GitHub will not let you click.

## The dataset

This session uses the UCI **Ionosphere** dataset (id 52) — 351 real radar returns from a
phased-array antenna in Goose Bay, Labrador, described by 34 continuous features (17
pulse readings, each as a complex number split into real and imaginary parts) and a
binary target: `g` ("good", the return shows structure in the ionosphere) or `b` ("bad",
the signal passed through).

It suits a quality gate well for three reasons. It is small enough that a full retrain in
a GitHub Actions runner takes a couple of seconds, so the gate can run on *every* PR
without anyone resenting it. It is genuinely learnable but not trivial — a reasonable
model lands around 0.92 accuracy, leaving real room both above and below, unlike a
dataset where everything scores 0.99 and no threshold discriminates. And its features are
opaque radar measurements with no human-readable meaning, which is exactly the situation
where a reviewer *cannot* eyeball a diff and judge whether a feature change was wise —
so the gate has to do it.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly what
to look at in the output; *Infer* says what it means, and what a different result would
have told you. For this session in particular, read the gate's *printed comparison table*
rather than just its exit code — a gate that passes for the wrong reason (a leaked
holdout, a resplit test set) looks identical from the outside to one that works.

## Prerequisites

```bash
pip install scikit-learn pandas ucimlrepo joblib
```

You also need a GitHub repository with **branch protection** enabled on `main`, requiring
the `model-quality-gate` check to pass. Without that setting the workflow still runs and
still reports failure — it just doesn't stop anyone merging, which makes it a suggestion
rather than a gate. The notebook below runs the gate logic locally so you can see both
outcomes without opening two real pull requests.

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

ionosphere = fetch_ucirepo(id=52)
X = ionosphere.data.features
y = ionosphere.data.targets["Class"]

print(f"{X.shape[0]} rows, {X.shape[1]} features")
print(y.value_counts())
print(X.describe().loc[["mean", "std"]].iloc[:, :4].round(3))

**Observe:** `351 rows, 34 features`, the class counts
(`g: 225, b: 126`), and the four-column summary — note that `Attribute2` has
**mean 0.000 and std 0.000**.
**Infer:** two facts shape the gate's thresholds. The classes are imbalanced about
64/36, so a model that predicts `g` for everything scores 0.64 accuracy — that is the
floor a naive model achieves, and it is why the absolute threshold is set at 0.88 rather
than somewhere near 0.5. And `Attribute2` is constant across the entire dataset: a
genuinely useless column that any reasonable model ignores. Keep it in mind — a
"cleanup" PR that removes constant columns *should* be a no-op on the metrics, and if
your gate reports a change from removing it, the gate itself is nondeterministic.

## Step 2 — Freeze the evaluation set

This is the most important step in the session and the one most often skipped.

If each CI run does its own `train_test_split`, the candidate and the deployed model are
scored on **different test sets**, and the difference between their numbers is partly
sampling noise. On 88 test rows, a two-sample swing moves accuracy by 2.3 points — enough
to block a good PR or wave a bad one through. A gate that fires at random gets disabled
within a week.

So: split once, write the holdout to disk, commit it, and hash it. Every model ever
compared is scored on those exact rows.

In [ ]:
import hashlib, json, os
from sklearn.model_selection import train_test_split

os.makedirs("data", exist_ok=True)

X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)
pd.concat([X_train, y_train], axis=1).to_csv("data/train.csv", index=False)
pd.concat([X_hold, y_hold], axis=1).to_csv("data/holdout.csv", index=False)

digest = hashlib.sha256(open("data/holdout.csv", "rb").read()).hexdigest()
print(f"train rows   : {len(X_train)}")
print(f"holdout rows : {len(X_hold)}  ({y_hold.value_counts().to_dict()})")
print(f"holdout sha256: {digest[:16]}...")

**Observe:** `train rows : 263`, `holdout rows : 88  ({'g': 56, 'b': 32})`,
and a 16-character hash prefix such as `9f2c41a8be07d3b5...`.
**Infer:** the stratified holdout preserves the 64/36 class ratio, so the accuracy floor
means the same thing on it as on the full dataset. The hash is not decoration — it goes
into the workflow as an assertion. If someone regenerates the holdout (deliberately or by
re-running this cell with a different seed), the hash changes, the gate refuses to run,
and you get a loud failure instead of a silent comparison between models scored on
different data. Silent is the dangerous outcome: every number still looks plausible.

## Step 3 — The training script

CI cannot run a notebook. Everything the gate needs lives in two small scripts that both
the pipeline and you can run identically from a terminal.

In [ ]:
%%writefile train.py
"""Train the candidate model. Reads data/train.csv, writes models/candidate.joblib."""
import json, os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

DROP_FEATURES: list[str] = []          # PRs edit this line -- see Step 7
N_ESTIMATORS = 300
MAX_DEPTH = None

def main() -> None:
    df = pd.read_csv("data/train.csv")
    y = df["Class"]
    X = df.drop(columns=["Class"] + DROP_FEATURES)

    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=0, n_jobs=-1
    )
    model.fit(X, y)

    os.makedirs("models", exist_ok=True)
    joblib.dump({"model": model, "features": X.columns.tolist()}, "models/candidate.joblib")
    print(json.dumps({"trained_on": len(X), "n_features": X.shape[1],
                      "estimator": type(model).__name__}))

if __name__ == "__main__":
    main()

**Observe:** `Writing train.py`.
**Infer:** note what the script persists: not just the fitted estimator but the
**feature list it was trained on**. The gate loads a candidate and a deployed model that
may have been trained on different columns; without the list travelling alongside each
model, scoring a model on the holdout means guessing at column order, and a silent
misalignment produces a number that is wrong rather than an error that is obvious. Also
note `random_state=0` — the gate compares *models*, so the training procedure itself must
be deterministic, otherwise a PR that changes nothing at all can still move the metrics.

## Step 4 — Record what is currently deployed

The regression half of the gate needs a reference. This file is the production model's
scorecard: it is written by the *promotion* step (Step 9), never by hand in normal
operation, and it is committed to the repo so the gate can read it without querying a
model registry.

Here we seed it with the metrics of the model already serving traffic.

In [ ]:
import json

deployed = {
    "model_id": "ionosphere-rf-v3",
    "promoted_at": "2026-07-14T09:22:41Z",
    "holdout_sha256_prefix": "9f2c41a8be07d3b5",
    "metrics": {"accuracy": 0.9205, "macro_f1": 0.9106},
}
with open("deployed_metrics.json", "w") as f:
    json.dump(deployed, f, indent=2)

print(json.dumps(deployed, indent=2))

**Observe:** the JSON, in particular
`"accuracy": 0.9205, "macro_f1": 0.9106` and the embedded
`holdout_sha256_prefix`.
**Infer:** the deployed model scores 0.9205 — comfortably above the 0.88 absolute floor.
That gap is the whole reason the regression rule exists: a candidate could score 0.89,
clear the floor, and still be two full points *worse* than what is live. Absolute
thresholds alone would ship it. The `holdout_sha256_prefix` field pins these numbers to
the specific evaluation data they were measured on, so the gate can refuse to compare
across a holdout change rather than producing a meaningless delta.

## Step 5 — The gate itself

Read the exit codes: `0` lets the PR merge, `1` blocks it. Everything else the script
prints is for the human reading the failed check.

The regression rule uses a small tolerance (`0.005`) rather than demanding strict
improvement. Requiring every PR to beat production means a refactor that legitimately
changes nothing gets blocked by floating-point noise; allowing a tolerance means only a
*meaningful* drop trips it. Setting the tolerance is a judgement call about how much
degradation you will accept in exchange for the change's other benefits.

In [ ]:
%%writefile quality_gate.py
"""Block the PR unless the candidate clears the floor AND doesn't regress."""
import hashlib, json, sys
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

MIN_ACCURACY = 0.88
MIN_MACRO_F1 = 0.90
REGRESSION_TOLERANCE = 0.005

def evaluate(path: str) -> dict:
    bundle = joblib.load(path)
    df = pd.read_csv("data/holdout.csv")
    y = df["Class"]
    X = df[bundle["features"]]
    pred = bundle["model"].predict(X)
    return {"accuracy": round(float(accuracy_score(y, pred)), 4),
            "macro_f1": round(float(f1_score(y, pred, average="macro")), 4)}

def main() -> int:
    deployed = json.load(open("deployed_metrics.json"))
    digest = hashlib.sha256(open("data/holdout.csv", "rb").read()).hexdigest()
    if not digest.startswith(deployed["holdout_sha256_prefix"]):
        print(f"::error::holdout changed ({digest[:16]}) -- metrics are not comparable")
        return 1

    cand = evaluate("models/candidate.joblib")
    prod = deployed["metrics"]
    failures = []

    print(f"{'metric':<12}{'candidate':>11}{'deployed':>11}{'floor':>9}{'delta':>9}")
    for name, floor in (("accuracy", MIN_ACCURACY), ("macro_f1", MIN_MACRO_F1)):
        delta = cand[name] - prod[name]
        print(f"{name:<12}{cand[name]:>11.4f}{prod[name]:>11.4f}{floor:>9.2f}{delta:>+9.4f}")
        if cand[name] < floor:
            failures.append(f"{name} {cand[name]:.4f} below floor {floor:.2f}")
        if delta < -REGRESSION_TOLERANCE:
            failures.append(f"{name} regressed {delta:+.4f} vs {deployed['model_id']}")

    json.dump(cand, open("candidate_metrics.json", "w"), indent=2)
    if failures:
        for f in failures:
            print(f"::error::QUALITY GATE FAILED -- {f}")
        return 1
    print("\nQUALITY GATE PASSED -- candidate is eligible for merge and promotion.")
    return 0

if __name__ == "__main__":
    sys.exit(main())

**Observe:** `Writing quality_gate.py`, and re-read the two rules in
`main()` — a *floor* check per metric and a *delta* check per metric, both appending to
the same `failures` list.
**Infer:** the script deliberately collects every failure rather than returning on the
first one. A PR author who fixes the accuracy problem only to discover an F1 problem on
the next CI run will start guessing; showing both at once means one round trip. The
`::error::` prefix is GitHub Actions' annotation syntax — those lines surface as
red inline annotations in the PR's Checks tab instead of being buried in log output that
nobody expands. And note the holdout-hash check runs *first* and returns before any
model is scored: if the evaluation data changed, there is no honest comparison to make
and reporting a delta anyway would be worse than failing.

## Step 6 — The GitHub Actions workflow

This is Session 10's workflow with one job added. The unit tests still run — the gate
supplements ordinary CI, it does not replace it. Note the job ordering: tests first, gate
second, and the gate only runs if the tests pass, because there is no point spending
runner minutes training a model from code that does not import.

In [ ]:
%%writefile .github/workflows/model-quality-gate.yml
name: model-quality-gate

on:
  pull_request:
    branches: [main]

jobs:
  tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt
      - run: pytest -q

  quality-gate:
    needs: tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt

      - name: Train candidate model
        run: python train.py

      - name: Evaluate against floor and deployed baseline
        run: python quality_gate.py | tee gate_report.txt

      - name: Comment gate result on the PR
        if: always()
        uses: actions/github-script@v7
        with:
          script: |
            const fs = require('fs');
            const body = '### Model quality gate\n```\n'
                       + fs.readFileSync('gate_report.txt', 'utf8') + '\n```';
            github.rest.issues.createComment({
              issue_number: context.issue.number,
              owner: context.repo.owner,
              repo: context.repo.repo,
              body,
            });

      - name: Upload candidate model
        if: success()
        uses: actions/upload-artifact@v4
        with:
          name: candidate-model
          path: |
            models/candidate.joblib
            candidate_metrics.json

**Observe:** `Writing .github/workflows/model-quality-gate.yml`, and
the three conditional keys — `needs: tests`, `if: always()` on the comment step, and
`if: success()` on the upload step.
**Infer:** those three lines encode the whole policy. `needs: tests` means broken code
never reaches training. `if: always()` on the PR comment is the one people get wrong: the
default is to skip subsequent steps after a failure, so without it the gate posts its
report only when it *passes* — precisely backwards, since the failing case is the one the
author needs to read. `if: success()` on the artifact upload means a rejected model is
never published anywhere; there is no half-approved binary sitting in artifact storage
waiting for someone to grab it by mistake.

For the failing check to actually block the merge button, add `quality-gate` to the
required status checks under **Settings → Branches → Branch protection rules**. Without
that, everything below still happens — GitHub just shows a red X above an enabled merge
button.

## Step 7 — PR #41: a change that passes the gate

> **PR #41 — "Increase forest size to 500 trees for stability"**
> Contributor changes `N_ESTIMATORS = 300` → `500` in `train.py`.

Simulating what the runner does: train from `data/train.csv`, then score against the
frozen holdout.

In [ ]:
# Simulating PR #41's train.py: N_ESTIMATORS bumped 300 -> 500
import joblib, os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

train_df = pd.read_csv("data/train.csv")
y_tr = train_df["Class"]
X_tr = train_df.drop(columns=["Class"])

model_a = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
os.makedirs("models", exist_ok=True)
joblib.dump({"model": model_a, "features": X_tr.columns.tolist()}, "models/candidate.joblib")
print(f"trained on {len(X_tr)} rows x {X_tr.shape[1]} features, 500 trees")

**Observe:** `trained on 263 rows x 34 features, 500 trees`.
**Infer:** the feature count is still 34 — this PR touched only a hyperparameter, so the
model's input contract is unchanged and the comparison against the deployed model is
apples-to-apples. Whenever this number differs from what the deployed model used, the
gate is comparing two structurally different models; that is legitimate (the whole point
is to allow improvements) but it means a passing gate is the *only* evidence the change
was good, so the numbers below carry more weight than usual.

In [ ]:
!python quality_gate.py; echo "exit code: $?"

**Observe:** the comparison table and the exit code:

```
metric        candidate   deployed    floor    delta
accuracy         0.9432     0.9205     0.88  +0.0227
macro_f1         0.9377     0.9106     0.90  +0.0271

QUALITY GATE PASSED -- candidate is eligible for merge and promotion.
exit code: 0
```

**Infer:** both metrics clear their floor and both deltas are positive, so neither rule
fires. Exit code `0` is what GitHub turns into a green check and an enabled merge button.
Read the deltas as well as the verdict, though: `+0.0227` accuracy on an 88-row holdout is
exactly two extra correct predictions. That is a pass, not a discovery — a sensible
reviewer merges this because it is harmless and slightly positive, not because 500 trees
have been proven better than 300. The gate answers "is this safe to ship", never "is this
an improvement worth celebrating".

## Step 8 — PR #42: a change the gate blocks

> **PR #42 — "Drop low-variance radar features to speed up training"**
> Contributor adds `DROP_FEATURES = ["Attribute" + str(i) for i in range(1, 10)]` to
> `train.py`, reasoning that the early pulse readings look noisy.

This is exactly the kind of change nobody can evaluate by reading the diff. The features
have no human-readable meaning, the justification sounds reasonable, unit tests pass, and
training gets measurably faster.

In [ ]:
# Simulating PR #42's train.py: first nine attributes dropped
DROP_FEATURES = [f"Attribute{i}" for i in range(1, 10)]

X_tr_b = X_tr.drop(columns=DROP_FEATURES)
model_b = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_tr_b, y_tr)
joblib.dump({"model": model_b, "features": X_tr_b.columns.tolist()}, "models/candidate.joblib")
print(f"trained on {len(X_tr_b)} rows x {X_tr_b.shape[1]} features (dropped {len(DROP_FEATURES)})")

**Observe:** `trained on 263 rows x 25 features (dropped 9)`, and note
that this cell **overwrites** `models/candidate.joblib`.
**Infer:** each CI run produces exactly one candidate at a fixed path, which is why the
gate never has to be told which model to evaluate. Locally, that also means Step 7's
model is now gone — if you want to compare the two candidates side by side, save them
under distinct names. The dropped features include `Attribute1` and `Attribute2`, one of
which is genuinely constant (Step 1) and contributes nothing; the other seven are not,
and the gate is about to price that difference.

In [ ]:
!python quality_gate.py; echo "exit code: $?"

**Observe:** the same table, a different verdict:

```
metric        candidate   deployed    floor    delta
accuracy         0.8636     0.9205     0.88  -0.0569
macro_f1         0.8503     0.9106     0.90  -0.0603
::error::QUALITY GATE FAILED -- accuracy 0.8636 below floor 0.88
::error::QUALITY GATE FAILED -- accuracy regressed -0.0569 vs ionosphere-rf-v3
::error::QUALITY GATE FAILED -- macro_f1 0.8503 below floor 0.90
::error::QUALITY GATE FAILED -- macro_f1 regressed -0.0603 vs ionosphere-rf-v3
exit code: 1
```

**Infer:** four annotations from two metrics — each tripped both rules, which is the
clearest possible signal that this is a real degradation rather than a threshold set too
tightly. Exit code `1` fails the job, the required check goes red, and branch protection
disables the merge button. Nobody had to know what `Attribute3` measures.

Now consider the near-miss version, because it is the case that justifies having two
rules. Had this candidate scored **0.8930**, the floor check would have passed (0.893 >
0.88) and only the regression check would have fired — a model that is "good enough" in
the absolute sense but five points worse than what is already serving traffic. Absolute
thresholds alone ship that model. The ratchet is what stops it.

In [ ]:
# What the contributor sees in the PR conversation, posted by the workflow's
# github-script step (if: always() -- which is why a FAILING gate still comments):
print("""### Model quality gate
```
metric        candidate   deployed    floor    delta
accuracy         0.8636     0.9205     0.88  -0.0569
macro_f1         0.8503     0.9106     0.90  -0.0603
```
""")
print("Checks: tests  PASS  (12 passed in 1.84s)")
print("Checks: quality-gate  FAIL  (Process completed with exit code 1)")
print("Merging is blocked: required status check 'quality-gate' has failed.")

**Observe:** the tests row says **PASS** while the gate row says
**FAIL**.
**Infer:** that combination is the entire argument for this session. PR #42 is correct
software — it imports, it runs, every unit test passes, and it is faster than what it
replaces. Session 10's pipeline would have shown a fully green PR and a reviewer would
have approved it in thirty seconds. The only thing standing between this change and
production is a check that retrained the model and compared a number to yesterday's
number. If you take one habit from this notebook, it is that ML CI has to assert on
*model behaviour*, because code-level tests are structurally incapable of seeing this.

## Step 9 — Promoting a passing candidate

A green gate makes a model *eligible*; it does not deploy it. Promotion is a separate
step on `main` after merge, and it does two things: ships the model, and — critically —
**raises the bar** by overwriting `deployed_metrics.json` with the new model's scores.

Skip that second part and the ratchet quietly stops working: the baseline stays frozen at
whatever the first model scored, and after a year of improvements the gate is comparing
every candidate to ancient history and waving through real regressions.

In [ ]:
import json, datetime, hashlib

# Re-train PR #41's accepted candidate and promote it
model_a = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
joblib.dump({"model": model_a, "features": X_tr.columns.tolist()}, "models/candidate.joblib")

candidate_metrics = {"accuracy": 0.9432, "macro_f1": 0.9377}   # from the passing gate run
digest = hashlib.sha256(open("data/holdout.csv", "rb").read()).hexdigest()

promoted = {
    "model_id": "ionosphere-rf-v4",
    "promoted_at": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "holdout_sha256_prefix": digest[:16],
    "metrics": candidate_metrics,
    "promoted_from_pr": 41,
}
json.dump(promoted, open("deployed_metrics.json", "w"), indent=2)
print(json.dumps(promoted, indent=2))

**Observe:** `"model_id": "ionosphere-rf-v4"`, the new
`"accuracy": 0.9432`, and `"promoted_from_pr": 41`.
**Infer:** the baseline moved from 0.9205 to 0.9432, so the next PR must now clear
0.9432 − 0.005 = **0.9382** rather than 0.9155. That is the ratchet turning one notch. A
PR that would have sailed through last week can legitimately be blocked this week, and
that is the system working as designed, not a bug — expect to explain it to a frustrated
contributor at least once.

`promoted_from_pr` is cheap provenance: months later, "why is production at 0.9432 and
where did that model come from" resolves to a single PR with its diff, review, and gate
report attached. In a larger setup this record lives in the MLflow model registry from
Session 1 instead of a JSON file, but the field that matters is the same one.

## When the gate itself is the problem

A gate that produces false failures gets bypassed with `--no-verify`, an admin merge, or
an "unblock the release" commit that removes the check — and once bypassing is normal, the
gate is decorative. Two ways it goes wrong, both worth recognizing by their signature.

### The flaky gate

**Symptom:** the same commit passes on one run and fails on the next. A contributor
re-runs the job until it goes green, which trains everyone that the gate is noise.

**Observe:** whether re-running the workflow on an *unchanged* commit produces
bit-identical numbers in the comparison table.
**Infer:** if the numbers move at all, something in the path is nondeterministic and the
gate cannot be trusted for deltas of the size it is trying to detect. The usual causes, in
order of frequency: a `train_test_split` without `random_state` somewhere (which is why
Step 2 freezes the holdout to disk rather than re-splitting in CI); an estimator without a
fixed `random_state`; unpinned dependencies, so a `scikit-learn` patch release changes a
tie-break; and thread-count differences between your laptop and the runner affecting
floating-point reduction order. The tolerance in Step 5 absorbs the last of these, not the
first three — do not widen it to paper over a real seed bug, or you will widen it right
past the regressions you built the gate to catch.

### The gate that passes on a leaked holdout

**Symptom:** every candidate scores suspiciously high — 0.99 accuracy where 0.92 was
normal — and every PR passes.

**Observe:** the row counts printed by `train.py` and the holdout hash check. If
`trained_on` suddenly reads `351` instead of `263`, training is consuming the whole
dataset, holdout included.
**Infer:** the gate is now measuring memorization and will approve anything. This happens
most often when someone "fixes" a data-loading path to read the original CSV instead of
`data/train.csv`. The tell is that scores jump *up* across the board rather than
fluctuating — a gate that suddenly becomes easy to pass deserves the same suspicion as
one that suddenly becomes impossible. The holdout-hash assertion in Step 5 catches a
*changed* holdout but not a *reused* one, so it is worth asserting on the training row
count explicitly as well.

## Where this fits against Session 10

| | Session 10 | Session 24 |
|---|---|---|
| Trigger | push / PR | PR into `main` |
| What runs | lint, unit tests | lint, unit tests, **plus a full retrain** |
| What it asserts | the code works | the *model* is good enough |
| Reference point | none | the currently deployed model's metrics |
| On failure | red check | red check **+ blocked merge** via branch protection |
| Side effect on merge | deploy | promote, and **raise the baseline** |

Session 10 is the right place to start — this workflow is that workflow with one job
appended. The conceptual jump is that CI now has to hold state across runs
(`deployed_metrics.json`) and evaluate against fixed data (`data/holdout.csv`), neither of
which ordinary software CI needs.

## What to try next

* Add a **fairness or slice gate** alongside the accuracy gate: split the holdout by a
  signal-strength band and require no slice to fall below 0.80. Aggregate metrics hide
  a model that got better overall by getting much worse on a minority of inputs.
* Replace the `deployed_metrics.json` file with a lookup against the MLflow model
  registry from Session 1, so the baseline comes from whatever is actually tagged
  `Production` rather than a file that could drift out of sync with reality.
* Chain this gate in front of Session 23's `bentoml build` step, so a container image is
  only ever produced from a model that passed — the gate decides, BentoML packages.
* Extend the promotion step (Step 9) into Session 14's automated retrain-and-deploy
  pipeline, and add the data-drift trigger from Session 17 so retraining starts on its
  own when the input distribution moves rather than only when someone opens a PR.
* Deliberately raise `MIN_MACRO_F1` to 0.95 and re-run Step 7's passing candidate. Watch
  a perfectly good PR get blocked, and use it to think about who in your team owns these
  numbers and how a threshold change itself gets reviewed.